<a href="https://colab.research.google.com/github/meizhong986/WhisperJAV/blob/main/notebook/WhisperJAV_kaggle_pass2_only.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# WhisperJAV Kaggle Pass2-Only Parallel Edition (v1.8.14+)

**Dual-GPU File-Parallel Workflow (Pass 2 Only)**

| Feature | Value |
|---------|-------|
| **Mode** | Parallel Files (2x T4 GPU) |
| **Input** | `/kaggle/input` (Dataset) |
| **Output** | `/kaggle/working/output` |
| **Pass** | Pass 2 Only |

---

### **Key Differences**
- **Single-Pass**: Only runs **Pass 2** (skips Pass 1 entirely)
- **File-Level Parallelism**: Each GPU processes a *different* file
- **No Merging**: Direct output from Pass 2
- **Higher Throughput**: Optimized for large datasets


In [ ]:
#@title Step 1: Pass 2 Configuration

folder_name = "WhisperJAV" #@param {type:"string"}
subtitle_language = "Japanese" #@param ["Japanese", "direct-to-english", "Chinese", "Korean", "Russian"]
pass2_pipeline = "stable_ts" #@param ["faster_whisper", "stable_ts", "qwen2_asr"]
pass2_sensitivity = "default" #@param ["low", "default", "high", "ultra"]
pass2_model = "base" #@param ["tiny", "base", "small", "medium", "large"]
pass2_speech_segmenter = "none" #@param ["none", "auditok", "silero_vad", "ten_vad"]
pass2_scene_detector = "none" #@param ["none", "histogram", "adaptive", "content"]
platform = "kaggle" #@param ["kaggle", "colab"]

WHISPERJAV_CONFIG = {
    "platform": platform,
    "folder_name": folder_name,
    "subtitle_language": subtitle_language,
    "pass2_pipeline": pass2_pipeline,
    "pass2_sensitivity": pass2_sensitivity,
    "pass2_model": pass2_model,
    "pass2_speech_segmenter": pass2_speech_segmenter,
    "pass2_scene_detector": pass2_scene_detector,
}

from IPython.display import HTML, display
html_table = "<table style='border:1px solid #ccc; border-collapse:collapse; width:100%;'>"
for k, v in WHISPERJAV_CONFIG.items():
    html_table += f"<tr><td style='border:1px solid #ccc; padding:8px;'><b>{k}</b></td><td style='border:1px solid #ccc; padding:8px;'>{v}</td></tr>"
html_table += "</table>"
display(HTML(html_table))
print("✓ Configuration Ready")

In [ ]:
#@title Step 2: Setup & Environment

import os, shutil, subprocess, sys, time, socket
from pathlib import Path

os.environ['MPLBACKEND'] = 'Agg'
os.environ['TORCH_HUB_TRUST_REPO'] = '1'

def section(title):
    print(f"\n{'---'*14}\n{title}\n{'---'*14}")

def status(msg, ok=True):
    print(f"{'✓' if ok else '✗'} {msg}")

section("PRE-FLIGHT CHECKS")

if "WHISPERJAV_CONFIG" not in globals():
    raise RuntimeError("Run Step 1 first!")

cfg = WHISPERJAV_CONFIG
PLATFORM = cfg['platform']
print(f"Platform: {PLATFORM.upper()}")
print(f"Python: {sys.version.split()[0]}")

gpu_check = subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader", shell=True, capture_output=True, text=True)
gpus = [line.strip() for line in gpu_check.stdout.splitlines() if line.strip()]
status(f"GPU(s): {len(gpus)}")

section("INSTALLING DEPENDENCIES")
install_start = time.time()

if PLATFORM == "kaggle":
    kaggle_cache = Path("/kaggle/working/.cache")
    kaggle_cache.mkdir(parents=True, exist_ok=True)
    kaggle_temp = Path("/kaggle/working/temp")
    kaggle_temp.mkdir(parents=True, exist_ok=True)
    
    os.environ["HF_HOME"] = str(kaggle_cache)
    os.environ["TRANSFORMERS_CACHE"] = str(kaggle_cache / "transformers")
    os.environ["TORCH_HOME"] = str(kaggle_cache / "torch")
    os.environ["TMPDIR"] = str(kaggle_temp)
    status(f"Cache: {kaggle_cache}")
    
    prefix = "sudo " if shutil.which("sudo") else ""
    subprocess.run(f"{prefix}apt-get update -qq && {prefix}apt-get install -y -qq ffmpeg", shell=True, check=True, capture_output=True)
    status("FFmpeg installed")
    
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"], check=False, capture_output=True)
    
    core_libs = [
        "tqdm", "numba>=0.60.0", "llvmlite>=0.46.0", "tiktoken", "soundfile", "auditok", "requests", "colorama", "regex",
        "numpy>=2.0.0", "scipy>=1.13.0", "librosa>=0.11.0",
        "pysrt", "srt", "aiofiles", "jsonschema", "Pillow", "pyloudnorm", "pydantic>=2,<3",
        "faster-whisper>=1.1.0", "transformers", "optimum", "accelerate", "huggingface-hub>=0.25.0",
        "ten-vad", "silero-vad>=6.0", "pydub",
        "modelscope>=1.20", "onnxruntime>=1.16.0", "addict", "simplejson", "sortedcontainers", "packaging"
    ]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + core_libs, check=True, capture_output=True)
    status("Core libraries installed")
    
    git_pkgs = [
        ("ffmpeg-python", "git+https://github.com/kkroening/ffmpeg-python.git"),
        ("Whisper", "git+https://github.com/openai/whisper.git@main"),
        ("Stable-TS", "git+https://github.com/meizhong986/stable-ts-fix-setup.git@main"),
        ("ClearVoice", "git+https://github.com/modelscope/ClearerVoice-Studio.git#subdirectory=clearvoice"),
        ("WhisperJAV", "git+https://github.com/meizhong986/WhisperJAV.git@main"),
    ]
    
    for name, url in git_pkgs:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", url], check=True, capture_output=True)
        status(f"{name} installed")

section("VERIFICATION")
result = subprocess.run([sys.executable, "-c", "import whisperjav; print(whisperjav.__version__)"], capture_output=True, text=True)
status(f"WhisperJAV {result.stdout.strip()} Ready")

print(f"\nSetup completed in {time.time() - install_start:.0f}s")
print("Go to Step 3 →")

In [ ]:
#@title Step 3: Parallel Execution (Pass 2 Only)

import os, shutil, sys, time, subprocess, torch
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

if "WHISPERJAV_CONFIG" not in globals():
    raise RuntimeError("Run Step 1 first!")

cfg = WHISPERJAV_CONFIG
PLATFORM = cfg['platform']

if PLATFORM == "kaggle":
    INPUT_DIR = Path("/kaggle/input")
    OUTPUT_DIR = Path("/kaggle/working/output")
    TEMP_DIR = Path("/tmp/whisperjav")
    try:
        DATASET_DIR = next(d for d in INPUT_DIR.iterdir() if d.is_dir())
    except StopIteration:
        DATASET_DIR = INPUT_DIR
elif PLATFORM == "colab":
    INPUT_DIR = Path(f"/content/drive/MyDrive/{cfg['folder_name']}")
    OUTPUT_DIR = INPUT_DIR
    TEMP_DIR = Path("/content/temp")
    DATASET_DIR = INPUT_DIR
else:
    INPUT_DIR = Path("./")
    OUTPUT_DIR = Path("./output")
    TEMP_DIR = Path("./temp")
    DATASET_DIR = INPUT_DIR

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TEMP_DIR.mkdir(parents=True, exist_ok=True)

VIDEO_EXTS = {'.mp4', '.mkv', '.avi', '.mov', '.wmv', '.flv', '.webm', '.m4v', '.mp3', '.wav', '.flac', '.m4a'}
videos = sorted([f for f in DATASET_DIR.rglob("*") if f.suffix.lower() in VIDEO_EXTS])

print(f"Found {len(videos)} media files")

ngpus = torch.cuda.device_count()
MODE = "PARALLEL" if ngpus >= 2 else "SEQUENTIAL"
MAX_WORKERS = min(2, ngpus)
print(f"Mode: {MODE} ({ngpus} GPU(s))\n")

def build_cmd_pass2(video_path, output_dir):
    pass_temp = TEMP_DIR / f"p2_{video_path.stem}"
    pass_temp.mkdir(parents=True, exist_ok=True)
    pass_out = output_dir / "pass2"
    pass_out.mkdir(parents=True, exist_ok=True)
    
    executable = cfg.get('whisperjav_cmd', sys.executable + " -m whisperjav.main")
    if str(executable).endswith('whisperjav'):
        cmd = [str(executable)]
    else:
        cmd = list(str(executable).split())

    cmd += [
        str(video_path),
        "--output-dir", str(pass_out),
        "--temp-dir", str(pass_temp),
        "--keep-temp",
        "--mode", cfg['pass2_pipeline'],
        "--sensitivity", cfg['pass2_sensitivity'],
        "--subs-language", "direct-to-english" if cfg['subtitle_language'] == 'direct-to-english' else "native",
    ]

    if cfg['pass2_model']:
        cmd += ["--model", cfg['pass2_model']]
    if cfg.get('pass2_speech_segmenter') and cfg['pass2_speech_segmenter'] != 'none':
        cmd += ["--speech-segmenter", cfg['pass2_speech_segmenter']]
    if cfg.get('pass2_scene_detector') and cfg['pass2_scene_detector'] != 'none':
        cmd += ["--scene-detection-method", cfg['pass2_scene_detector']]

    return cmd, pass_out

def run_pass2(video_idx, video, gpu_id):
    cmd, out_dir = build_cmd_pass2(video, OUTPUT_DIR)
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(gpu_id % ngpus)
    
    t0 = time.time()
    try:
        proc = subprocess.run(cmd, env=env, capture_output=True, text=True, timeout=7200)
        success = proc.returncode == 0
    except subprocess.TimeoutExpired:
        success = False
    
    found_srt = list(out_dir.glob(f"{video.stem}*.srt"))
    result_srt = found_srt[0] if found_srt else None
    
    return {
        "idx": video_idx,
        "file": video.name,
        "success": success and (result_srt is not None),
        "srt": result_srt,
        "time": time.time() - t0,
        "gpu": gpu_id % ngpus
    }

RESULTS = []

if MODE == "PARALLEL" and len(videos) > 0:
    print("Processing files in parallel...\n")
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {}
        for i, video in enumerate(videos):
            future = executor.submit(run_pass2, i, video, i % ngpus)
            futures[future] = i
        
        for future in as_completed(futures):
            result = future.result()
            print(f"[{result['idx']+1}/{len(videos)}] {result['file']} (GPU {result['gpu']})")
            print(f"  Time: {result['time']:.0f}s | Status: {'✓ OK' if result['success'] else '✗ FAIL'}")
            
            if result['success']:
                final_srt = OUTPUT_DIR / result['srt'].name
                shutil.copy(result['srt'], final_srt)
                print(f"  Output: {final_srt.name}")
                RESULTS.append(final_srt)
            print()
else:
    print("Processing files sequentially...\n")
    for i, video in enumerate(videos):
        result = run_pass2(i, video, 0)
        print(f"[{i+1}/{len(videos)}] {result['file']}")
        print(f"  Time: {result['time']:.0f}s | Status: {'✓ OK' if result['success'] else '✗ FAIL'}")
        if result['success']:
            final_srt = OUTPUT_DIR / result['srt'].name
            shutil.copy(result['srt'], final_srt)
            print(f"  Output: {final_srt.name}")
            RESULTS.append(final_srt)
        print()

print(f"\n{'='*50}")
print(f"Processed {len(RESULTS)}/{len(videos)} files successfully")
print(f"{'='*50}")

if PLATFORM == "kaggle" and len(RESULTS) > 0:
    shutil.make_archive("/kaggle/working/subtitles", 'zip', OUTPUT_DIR)
    print("\n📦 Output zipped: /kaggle/working/subtitles.zip")